<img src="../assets/agent_patterns_02_overview.png" alt="Agent Loop와 Tool Calling 흐름" style="display:block;width:100%;max-width:1200px;margin:0 auto 24px;border-radius:12px;">


![Evaluated LLM-Orchestrated Food Workflow](../assets/workflow_diagrams/agent_loop_mcp.svg)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: 02 독립 Agent Loop + Tool Calling</h2>
<p style="margin: 7px 0;">02 전용 대화창·상태 카드·세로 타임라인을 사용합니다. Tool이 실제로 선택되면 해당 Agent 카드에서 종류, 입력, 읽기 전용 결과를 확인하고, Evaluation Middleware의 승인·보완 판정을 확인합니다. 보완은 최대 3회입니다.</p>
  </div>
</div>


In [2]:
from pathlib import Path
import importlib
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '4_agent_patterns' / 'src').is_dir())
if str(project_root / '4_agent_patterns') not in sys.path:
    sys.path.insert(0, str(project_root / '4_agent_patterns'))

import src.agent_loop_mcp.ui as loop_ui_module
loop_ui_module = importlib.reload(loop_ui_module)
if 'loop_ui' in globals():
    loop_ui.close()
loop_ui = loop_ui_module.launch_loop_ui()


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [3]:
# Marketing HTML Agent가 생성한 문서 미리보기
from html import escape
from src.agent_loop_mcp.tools import latest_marketing_html_draft

latest_ui_draft = latest_marketing_html_draft()
graph_tool_calls = result.get('tool_calls', []) if 'result' in globals() else []
html_draft = (latest_ui_draft or {}).get('html') or next(
    (call.get('result', {}).get('html') for call in graph_tool_calls
     if call.get('name') == 'generate_marketing_html'),
    None,
)
if html_draft:
    display(HTML(f'<iframe title="Marketing HTML preview" sandbox="" style="width:100%;height:420px;border:1px solid #d9e2f3;border-radius:8px;background:white" srcdoc="{escape(html_draft, quote=True)}"></iframe>'))
else:
    print('같은 노트북 커널에서 UI 또는 Graph 실행으로 생성된 Marketing HTML이 없습니다.')


같은 노트북 커널에서 UI 또는 Graph 실행으로 생성된 Marketing HTML이 없습니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>02 Agent Loop의 실행과 Agent별 스트리밍 카드를 확인한다.</li><li>Tool을 선택한 Agent의 완료 행과 카드 상세, Evaluation Middleware의 Tool audit·보완 판정을 확인한다.</li><li>외부 쓰기 Tool이 없고 보완 평가는 최대 3회임을 확인한다.</li></ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">노트북은 구현을 복사하지 않고 공개 코어 API만 호출합니다.</p>
  </div>
</div>


In [4]:
from src.agent_loop_mcp.lab_core import (
    list_food_mcp_tools,
    run_food_mcp_loop_steps,
)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Agent별 Tool 계약 확인</h2><p style="margin: 7px 0;">02 Agent Loop의 각 전문 Agent에 할당된 Function, API, Agent-as-a-Tool 계약을 공개 API로 확인합니다.</p></div></div>


In [5]:
tool_catalog = list_food_mcp_tools()
for tool in tool_catalog:
    print(f"{tool['agent_label']} | {tool['name']} ({tool['type']}) | read_only={not tool['write']}")
    print(f"  └ {tool['description']}")


Food R&D Agent | open_food_facts_search (API / read-only) | read_only=True
  └ Open Food Facts 공개 API에서 제품 키워드의 식품 라벨 데이터를 읽기 전용으로 조회합니다.
Marketing Agent | open_food_facts_search (API / read-only) | read_only=True
  └ Open Food Facts 공개 API에서 제품 키워드의 식품 라벨 데이터를 읽기 전용으로 조회합니다.
Marketing Agent | generate_marketing_image (Agent as a Tool) | read_only=True
  └ Marketing Image Agent에게 프로모션 HTML에 넣을 제품 이미지를 생성하도록 요청합니다. HTML 시안에 이미지를 넣어야 하면 generate_marketing_html과 함께 선택하세요.
Marketing Agent | generate_marketing_html (Agent as a Tool) | read_only=True
  └ Marketing HTML Agent에게 상품의 한 장짜리 프로모션 HTML 시안 생성을 요청합니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. Cell 단위 Agent Loop 실행</h2><p style="margin: 7px 0;">Observer, Router, Planning Agent, 실행 계획, 다음 Agent를 각각 별도 셀에서 진행합니다. <code>다음 Agent 또는 평가 1회 진행</code> 셀은 실행할 때마다 정확히 한 Agent의 Tool audit 또는 평가 단위 하나만 진행합니다.</p></div></div>


In [6]:
request = '고단백 닭가슴살 도시락의 원가·영양·식품안전·공급망을 검토하고 출시 권고안을 작성해줘.'
loop_steps = run_food_mcp_loop_steps(request)
result = None
print('Agent Loop session을 준비했습니다. 다음 셀부터 한 단위씩 진행하세요.')


Agent Loop session을 준비했습니다. 다음 셀부터 한 단위씩 진행하세요.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. Observer와 Router</h2><p style="margin: 7px 0;">두 셀에서 대화 관찰 결과와 routing score를 각각 확인합니다.</p></div></div>


In [7]:
observer_step = next(loop_steps)
assert observer_step['step'] == 'observer', observer_step['step']
print(observer_step['observation'])


{'use_history': False, 'observation': '', 'external_data_needed': False, 'needs_work': False}


In [8]:
route_step = next(loop_steps)
assert route_step['step'] == 'route', route_step['step']
print(route_step['decision'])


{'selected': 'work', 'scores': {'work': 0.76461047, 'chat': 0.31878194}, 'margin': 0.44582853000000006, 'engine': 'btp_cohere_rerank', 'low_confidence': False}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Planning Agent와 실행 계획</h2><p style="margin: 7px 0;">Planning Agent의 공통 브리프와 LLM이 선택한 단계별 Agent를 분리해 확인합니다.</p></div></div>


In [9]:
planning_step = next(loop_steps)
assert planning_step['step'] == 'agent', planning_step['step']
print(f"{planning_step['agent_label']} ({planning_step['agent_id']})")
print(planning_step['answer'])


Work Planning Agents (Orchestrator) (work-planning)
검증된 사실: 고단백 닭가슴살 도시락의 원가·영양·식품안전·공급망 검토와 출시 권고가 목표이며, 활용 가능한 전문가는 5개 분야다. 가정: 판매가·중량·목표 영양성분·생산량·유통기한·판매채널은 미정이므로 각 Agent가 합리적 기준을 설정하고 추가 확인사항을 명시한다.


In [10]:
plan_step = next(loop_steps)
assert plan_step['step'] == 'plan', plan_step['step']
for number, stage in enumerate(plan_step['plan']['stages'], start=1):
    print(f"{number}. {', '.join(stage['agent_ids'])}: {stage['objective']}")
print('최종 Agent:', plan_step['plan']['final_agent_id'])


1. food-rnd, procurement, quality-assurance: 제품 규격을 가정해 레시피·영양 목표·제조공정, 원재료 소요량·공급업체 기준·공급망 위험, HACCP·알레르기·미생물 및 품질관리 요구사항을 각각 검토한다.
2. marketing: 검토 결과와 목표 고객을 바탕으로 상품 콘셉트, 차별화 메시지, 표시·광고 유의사항, 판매채널 및 출시 전략을 제안한다.
3. finance: 앞선 결과를 반영해 원가·판매수익·공헌이익·손익분기점과 주요 재무 가정을 정리하고, 출시 여부·조건·우선 실행과제를 포함한 최종 권고안을 작성한다.
4. synthesis: 선행 전문 Agent 결과를 종합해 현재 사용자 요청에 대한 최종 답변을 작성합니다.
최종 Agent: synthesis


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. Agent Loop 자동 진행</h2><p style="margin: 7px 0;">이 셀을 한 번 실행하면 다음 Agent의 응답·Tool audit 또는 Evaluation Middleware 단위를 연속으로 진행하고, <code>complete</code>에서 멈춥니다.</p></div></div>


In [11]:
while result is None:
    next_step = next(loop_steps)
    if next_step['step'] == 'agent':
        phase = '보완' if next_step['rework'] else '실행'
        print(f"[{phase}] {next_step['agent_label']} ({next_step['agent_id']})")
        print(next_step['answer'])
        for call in next_step['tool_calls']:
            tool_result = call.get('result', {})
            boundary = tool_result.get('execution_boundary') if isinstance(tool_result, dict) else None
            print(f"  [Tool] {call['name']} | arguments={call.get('arguments', {})} | boundary={boundary or '읽기 전용'}")
    elif next_step['step'] == 'evaluation':
        review = next_step['evaluation']
        print(f"[Evaluation {review['loop']}] {review['verdict']}: {review['reason']}")
    elif next_step['step'] == 'complete':
        result = next_step['result']
        print(f"[complete] tools={len(result['tool_calls'])} | review_required={result['review_required']}")
        print(result['result'])
        break
    else:
        print(next_step)



[실행] Food R&D Agent (food-rnd)
**출시 권고(예비안·추정치)**  
규격: 450g/팩, 냉장 5일, 판매가 8,900원 가정. 닭가슴살 180g, 현미밥 150g, 채소 100g, 소스 20g. 예상 영양은 500~550kcal, 단백질 45~50g, 나트륨 800~1,000mg(실측 필요).  
원가 추정: 원료 2,200~2,800원, 포장·가공·물류 1,500~2,000원; 총 제조원가 3,700~4,800원.  
공정: 입고검사→냉장보관→가열(중심 75℃ 이상)→급속냉각→충전→금속검출→0~5℃ 유통. HACCP CCP는 가열·냉각·보관온도다. 닭·대두·밀 등 알레르기 교차오염을 관리한다.  
국산/대체 공급처 2곳 이상, 냉장 운송 이중화, 원료 로트 추적을 권장한다. 출시 전 미생물·유통기한·영양성분·관능 검증 및 실제 견적 확인이 필수다.
3초 후 다음 단계를 진행합니다…
[실행] Procurement Agent (procurement)
**출시 검토(가정: 450g·냉장 5일·8,900원/팩)**  
- 구성: 닭가슴살 180g, 현미밥 150g, 채소 100g, 소스 20g. 목표 500~550kcal, 단백질 45~50g, 나트륨 800~1,000mg(모두 추정, 실측 필요).  
- **추정 원가**: 원료 2,200~2,800원+가공·포장·물류 1,500~2,000원=3,700~4,800원. 실제 공급견적·수율 확인 필수.  
- HACCP: 가열 중심 75℃ 이상, 신속냉각, 0~5℃ 유지, 금속검출·로트추적. 닭·대두·밀 교차오염 및 미생물·유통기한 검증 필요.  
- 공급사는 HACCP, 안정 수율·냉장배송, 2개 이상 이원화. 냉동 닭·잡곡/계절채소 대체로 가격 변동 대응.  
- 채널·생산량·목표 마진·원산지·유통기한 확정 후, 파일럿 관능/영양/미생물 시험을 통과하면 조건부 출시 권고.
3초 후 다음 단계를 진행합니다…
[실행] Quality Assurance Agent (quality-assurance

KeyboardInterrupt: 